# Camada Bronze

Carrega os oito CSVs fornecidos pela V-Credit, a partir do volume `landing`, em tabelas Delta do schema `bronze`. Os dados entram sem nenhum tratamento, apenas com a coluna `ingestion_datetime`, que registra o momento da carga. A gravação é feita em modo append: cada execução acrescenta uma nova carga e preserva as anteriores. As Silvers ficam com a versão mais recente de cada registro.

## 1. Configuração

Define o catálogo, o schema e o caminho do volume, e associa cada tabela ao encoding do seu CSV. Os arquivos foram extraídos de sistemas diferentes: `chamados_hora` e `clientes` estão em ISO-8859-1, e os demais em UTF-8.

In [0]:
from pyspark.sql import functions as F

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
volume_path = f"/Volumes/{catalog}/bronze/landing"

# Dicionário entre tabela e sua respectiva codificação para ingestão na camada Bronze a partir dos CSVs
tabelas = {"chamados_hora": "ISO8859-1", 
           "base_atendentes": "UTF-8",
           "base_motivos": "UTF-8",
           "canais": "UTF-8",
           "chamados": "UTF-8", 
           "clientes": "ISO8859-1",
           "custos": "UTF-8",
           "pesquisa_satisfacao": "UTF-8"}



## 2. Limpeza das tabelas (opcional)

Como a gravação é em append, as cargas anteriores permanecem nas tabelas. Para recomeçar o histórico, por exemplo depois de corrigir um erro de leitura, basta descomentar e executar esta célula antes da ingestão.

In [0]:
# TRUNCATE TABLE caso desejado
# for t,_ in tabelas.items():
#     spark.sql(f"TRUNCATE TABLE {bronze_schema}.{t}")
#     print(f"Tabela {bronze_schema}.{t} truncada com sucesso.")

## 3. Ingestão

Cada CSV é lido com cabeçalho, com suporte a campos de várias linhas e a aspas escapadas por aspas duplas, e recebe a coluna `ingestion_datetime`.

O padrão de leitura `*{tabela}.csv` exige que o nome do arquivo termine exatamente com o nome da tabela. Um padrão mais aberto, como `*{tabela}*.csv`, fazia a leitura de `chamados` incluir também o arquivo `chamados_hora.csv`. Como o Spark junta CSVs pela posição das colunas, os horários de `chamados_hora` iam parar nas colunas `motivo`, `canal` e `resolvido`.

In [0]:
# Loop de criação das tabelas
for tabela, codificacao in tabelas.items():
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .option("encoding", codificacao)
          .csv(f"{volume_path}/*{tabela}.csv")
          .withColumn("ingestion_datetime", F.current_timestamp()))


    (df.write     # escreve a tabela como delta em modo append
       .format("delta")
       .mode("append")
       .saveAsTable(f"{bronze_schema}.{tabela}"))

## 4. Conferência da carga

Mostra o número de linhas de cada tabela e exibe o conteúdo de todas elas, como apoio para os tratamentos da Silver.

In [0]:
#numero de rows em cada tabela
tabelas = spark.catalog.listTables(bronze_schema)
for t in tabelas:
  print(f"{t.name}: {spark.table(f'{bronze_schema}.{t.name}').count()} linhas")

In [0]:
# Celula adicional mostrando todas as tabelas da bronze carregadas (serve de apoio para as transformações feitas na camada silver)

for t in tabelas:
    display(spark.table(f'{bronze_schema}.{t.name}'))